In [52]:
from transformers import BertTokenizerFast, AutoModelForCausalLM
import torch
import numpy as np

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
model = AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese")

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 50954.04it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ckiplab/gpt2-base-chinese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [163]:
SPECIAL_TOKENS = "[PAD]", "[CLS]", "[SEP]"
SPECIAL_TOKEN_IDS = 0, 101, 102  # BERT token IDs: 0 = [PAD]; 101 = [CLS]; 102 = [SEP]
OTHER_TOKENS = "[UNK]", "##"

MAX_LENGTH = 256
BATCH_SIZE =208

def get_word_logprobs(*, tokens: list, split_sentences: list,
                      ids: np.ndarray, token_logprobs: np.ndarray,
                      special_tokens: tuple | list | None=None,
                      special_token_ids: tuple| list| None=None,
                      other_tokens: tuple | list | None=None) -> np.ndarray:
    if special_tokens is None or special_token_ids is None:   
        special_tokens = SPECIAL_TOKENS
        special_token_ids = SPECIAL_TOKEN_IDS
    if other_tokens is None:
        other_tokens = OTHER_TOKENS

    tokens_no_special = [list(filter(lambda s: s not in special_tokens, t)) for t in tokens]
    
    spans = [[0] for _ in tokens_no_special]
    for i, (trow, srow) in enumerate(zip(tokens_no_special, split_sentences)):
        chunk = ""
        idx = 0
        for j, t in enumerate(trow):
            if t in other_tokens:
                print(f"{t} found in sentence index {i}, token index {j}:\n  {trow}")
            if srow[idx].startswith(chunk + t):
                chunk += t
            else:
                chunk = t
                idx += 1
                spans[i].append(j)
        spans[i].append(len(tokens_no_special[i]))

    check_span = [["".join(trow[seq[i]: seq[i+1]]) for i in range(len(seq)-1)]
                                          for trow, seq in zip(tokens_no_special, spans)]
    aligned = np.array([check == split for check, split in zip(check_span, split_sentences)])
    not_aligned_where = np.where(aligned == False)[0]
    if not_aligned_where.size != 0:
        print(f"sentence indices where alignment failed: {not_aligned_where}")
    
    token_logprobs_masked = [tprobs[~ np.isin(_id, special_token_ids)] for _id, tprobs in zip(ids, token_logprobs)]
    word_logprobs = [[sum(tprobs[seq[i]: seq[i+1]]) for i in range(len(seq)-1)]
                                                    for tprobs, seq in zip(token_logprobs_masked, spans)]
    return np.array(word_logprobs)

def get_llm_token_logprobs(sentences: list,
                       tokenizer: BertTokenizerFast, model: AutoModelForCausalLM,
                       max_length: int=256) -> tuple:
    inputs = tokenizer(sentences,
                   truncation=True,
                   padding=True,
                   return_tensors="pt",
                   return_offsets_mapping=False,
                   max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    ids = inputs["input_ids"]      
    tokens = [tokenizer.convert_ids_to_tokens(i) for i in ids]

    with torch.inference_mode():
        outputs = model(**inputs,
                        output_hidden_states=False)
    logits = outputs.logits    # shape = (batch_size, seq_len, vocab_size)
    logprobs = torch.log_softmax(logits, dim=-1)

    target_ids = ids[:, 1:]
    token_logprobs = torch.gather(logprobs[:, :-1, :],
                                  dim=-1,
                                  index=target_ids.unsqueeze(-1)).squeeze(-1)
    token_logprobs = torch.cat([torch.zeros((target_ids.size(0), 1), device=token_logprobs.device),
                                token_logprobs],
                                dim=1)
    token_logprobs = token_logprobs.detach().cpu().numpy()
    # token_logprobs = np.array([[logprobs[i, j, idx] for j, idx in enumerate(target_ids[i])]
                                       # for i in range(target_ids.size(0))]) <- works, but too many for loops
    ids = ids.detach().cpu().numpy()

    return ids, tokens, token_logprobs


sentences = ["這對夫妻攜手走過四十年現在依然非常",
        "他們依偎彼此手牽手漫步在沙灘上看來非常"]
split_sentences = [["這對", "夫妻", "攜手", "走過", "四十年", "現在","依然","非常"],
        ["他們", "依偎", "彼此", "手牽手", "漫步在", "沙灘上", "看來" ,"非常"]]

ids, tokens, token_logprobs = get_llm_token_logprobs(sentences,
                                                     tokenizer,
                                                     model,
                                                     max_length=MAX_LENGTH)
word_logprobs = get_word_logprobs(tokens=tokens,
                                  split_sentences=split_sentences,
                                  ids=ids,
                                  token_logprobs=token_logprobs,
                                  special_tokens=SPECIAL_TOKENS,
                                  special_token_ids=SPECIAL_TOKEN_IDS,
                                  other_tokens=OTHER_TOKENS)
print(word_logprobs)

[[-10.790742   -6.398704   -8.881091   -7.1071763  -8.212315   -9.810731
   -7.7474246  -5.378808 ]
 [ -7.9810777 -13.73241    -9.133151  -16.962288  -18.265514   -7.4629636
  -10.648433   -5.0953746]]
